# MOAB Rover Survey Lab: Feature Engineering in Snowflake

## Goal
In this lab, we will engineer new features from rover-collected survey data using:
- Python UDFs
- SQL views
- AI prompts

## Raw Inputs
- Easting
- Northing
- Sensor measurement

## Engineered Features
1. Grid tile, survey unit, and subcell assignment
2. Tile-level aggregated measurement signals
3. Comparison to normal range
4. Remediation prioritization

## SCALARS Mapping
- **Simplify**: convert coordinates into tile IDs
- **Aggregate**: summarize measurements at the tile level using multi-level aggregation
- **Assess**: compare readings to expected range
- **Rank / Score**: prioritize areas for remediation

In [ ]:
%%sql -r dataframe_1
SELECT
  *
FROM
  data5035.spring26.sdg_001_ra226_scandata 
LIMIT
  10;

## Convert from Coordinates to Grid

The input data is provided in directional distances on a flat map projection. Northing and Easting indicate how far to go in those directions (up and right) in US Feet relative to a known starting point. Our purpose here is to map those directional distances on to a three-level grid.

### Measurement
* **Tiles** are the largest areas. They are composed of a grid of **Survey Units** 21 tiles wide x 18 tiles tall.
* **Survey Units** measure 32.81 ft x 32.81 ft square
* **Subcells** are square subdivisions within the **Survey Units** laid out 10x10

### Labeling
* **Tiles** are coded by row letter and column letter starting at AA in the bottom-left of our map, given some known origin (2180160.001, 6660000.000). AA indicates 1st row, 1st column. AB indicates 1st row, 2nd column to the left. BA indicates 2nd row up, 1st column.
* Within each Tile, **Survey Units** are numbered starting in the top-left corner, proceeeding right, then down to the beginning of the next row (as if you're reading down a page)
* Within each Survey Unit, **Subcells** are numbered starting in the bottom-left corner, proceeduing right, then up to the beginning of thext row (as if you're reading from the bottom of a page up)

**Create a Python UDF to convert x (easting), y (northing) into the grid labels.**

### Testing

```
    >>> convert_xy(2180160.0001, 6660000.0000)  
    ('AA', 358, 1)

    >>> convert_xy(2180160.0001 + 32.81*21.01, 6660000.0000)
    ('AB', 358, 1)

    >>> convert_xy(2180160.0001 + 32.81*22.01 + 4, 6660000.0000 + 32.81*19.01 + 4)
    ('BB', 338, 12)
```

In [ ]:
CREATE OR REPLACE FUNCTION CONVERT_XY(
        X FLOAT,
        Y FLOAT,
        ORIGIN_X FLOAT,
        ORIGIN_Y FLOAT,
        SU_SIZE FLOAT,
        TILE_GRID_X NUMBER(38,0),
        TILE_GRID_Y NUMBER(38,0),
        SUBCELL_GRID NUMBER(38,0)
    )
    RETURNS OBJECT
    LANGUAGE PYTHON
    RUNTIME_VERSION = '3.11'
    HANDLER = 'CONVERT_XY'
    AS 
$$
import math

def CONVERT_XY(X, Y, ORIGIN_X, ORIGIN_Y, SU_SIZE, TILE_GRID_X, TILE_GRID_Y, SUBCELL_GRID):
    dx = X - ORIGIN_X
    dy = Y - ORIGIN_Y

    su_x = dx / SU_SIZE
    su_y = dy / SU_SIZE

    tile_col = int(math.floor(su_x / TILE_GRID_X))
    tile_row = int(math.floor(su_y / TILE_GRID_Y))

    local_su_x = su_x - tile_col * TILE_GRID_X
    local_su_y = su_y - tile_row * TILE_GRID_Y

    su_col = int(math.floor(local_su_x))
    su_row = int(math.floor(local_su_y))

    su_row_from_top = (TILE_GRID_Y - 1) - su_row
    su_number = su_row_from_top * TILE_GRID_X + su_col + 1

    frac_x = local_su_x - su_col
    frac_y = local_su_y - su_row

    subcell_col = int(math.floor(frac_x * SUBCELL_GRID))
    subcell_row = int(math.floor(frac_y * SUBCELL_GRID))

    subcell_number = subcell_row * SUBCELL_GRID + subcell_col + 1

    row_letter = chr(ord('A') + tile_row)
    col_letter = chr(ord('A') + tile_col)
    tile_label = row_letter + col_letter

    return {"tile": tile_label, "su": su_number, "subcell": subcell_number}
$$;

In [ ]:
CREATE OR REPLACE FUNCTION CONVERT_XY(X FLOAT, Y FLOAT)
    RETURNS OBJECT
    LANGUAGE SQL
    AS
    $$
        SELECT DATA5035.DOLPHIN.CONVERT_XY(
            X, Y,
            2180160.0001,
            6660000.0000,
            32.81,
            21,
            18,
            10
        )
    $$;

In [ ]:
%%sql -r dataframe_4
select convert_xy(2180160.0001, 6660000.0000);

In [ ]:
%%sql -r dataframe_5
select convert_xy(2180160.0001 + 32.81*21.01, 6660000.0000);

In [ ]:
%%sql -r dataframe_6
select convert_xy(2180160.0001 + 32.81*22.01 + 4, 6660000.0000 + 32.81*19.01 + 4)

In [ ]:
SELECT 
    convert_xy(easting, northing) AS coordinates, 
    coordinates:su::INTEGER AS su,
    coordinates:subcell::INTEGER AS subcell,
    coordinates:tile::STRING AS tile,
    * 
FROM 
    data5035.spring26.sdg_001_ra226_scandata 
LIMIT 100;

## Compute Layered Averages

In [ ]:
WITH grid AS (
    SELECT
        CONVERT_XY(EASTING, NORTHING) AS coordinates,
        coordinates:tile::STRING AS tile,
        coordinates:su::INTEGER AS su,
        coordinates:subcell::INTEGER AS subcell,
        READING
    FROM data5035.spring26.sdg_001_ra226_scandata
),
subcell_avg AS (
    SELECT
        tile,
        su,
        subcell,
        AVG(READING) AS avg_reading
    FROM grid
    GROUP BY tile, su, subcell
),
su_avg AS (
    SELECT
        tile,
        su,
        AVG(avg_reading) AS avg_reading
    FROM subcell_avg
    GROUP BY tile, su
),
tile_avg AS (
    SELECT
        tile,
        AVG(avg_reading) AS avg_reading
    FROM su_avg
    GROUP BY tile
)
SELECT * FROM tile_avg
ORDER BY tile;

## Compare to Reference Ranges

This measurement is of Radium-226 levels.
* `<5` - OK
* `5 <= X < 7.4` - Warning
* `>= 7.4` - Alarm

In [ ]:
WITH grid AS (
    SELECT
        CONVERT_XY(EASTING, NORTHING) AS coordinates,
        coordinates:tile::STRING AS tile,
        coordinates:su::INTEGER AS su,
        coordinates:subcell::INTEGER AS subcell,
        READING
    FROM data5035.spring26.sdg_001_ra226_scandata
),
subcell_avg AS (
    SELECT
        tile,
        su,
        subcell,
        AVG(READING) AS avg_reading
    FROM grid
    GROUP BY tile, su, subcell
),
su_avg AS (
    SELECT
        tile,
        su,
        AVG(avg_reading) AS avg_reading
    FROM subcell_avg
    GROUP BY tile, su
),
tile_avg AS (
    SELECT
        tile,
        AVG(avg_reading) AS avg_reading
    FROM su_avg
    GROUP BY tile
)
SELECT
    tile,
    ROUND(avg_reading, 4) AS avg_reading,
    CASE
        WHEN avg_reading < 5 THEN 'OK'
        WHEN avg_reading < 7.4 THEN 'Warning'
        ELSE 'Alarm'
    END AS status
FROM tile_avg
ORDER BY tile;

## Prioritize

Once we've build everything out, let's use AI to make some recommendations on remediation priorities.

In [ ]:
WITH grid AS (
    SELECT
        CONVERT_XY(EASTING, NORTHING) AS coordinates,
        coordinates:tile::STRING AS tile,
        coordinates:su::INTEGER AS su,
        coordinates:subcell::INTEGER AS subcell,
        READING
    FROM data5035.spring26.sdg_001_ra226_scandata
),
subcell_avg AS (
    SELECT
        tile,
        su,
        subcell,
        AVG(READING) AS avg_reading
    FROM grid
    GROUP BY tile, su, subcell
),
su_avg AS (
    SELECT
        tile,
        su,
        AVG(avg_reading) AS avg_reading
    FROM subcell_avg
    GROUP BY tile, su
),
tile_avg AS (
    SELECT
        tile,
        AVG(avg_reading) AS avg_reading
    FROM su_avg
    GROUP BY tile
),
classified AS (
    SELECT
        tile,
        ROUND(avg_reading, 4) AS avg_reading,
        CASE
            WHEN avg_reading < 5 THEN 'OK'
            WHEN avg_reading < 7.4 THEN 'Warning'
            ELSE 'Alarm'
        END AS status
    FROM tile_avg
)
SELECT SNOWFLAKE.CORTEX.COMPLETE(
    'llama3.1-70b',
    'You are an environmental remediation specialist reviewing Radium-226 survey data. ' ||
    'Reference ranges: <5 pCi/g = OK, 5-7.4 pCi/g = Warning, >=7.4 pCi/g = Alarm. ' ||
    'Here are the tile-level layered average results:\n\n' ||
    LISTAGG('Tile ' || tile || ': ' || avg_reading || ' pCi/g (' || status || ')', '\n')
        WITHIN GROUP (ORDER BY avg_reading DESC) ||
    '\n\nProvide a prioritized remediation plan with specific recommendations for each status category. Be concise.'
) AS remediation_priorities
FROM classified;